In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
import time
import re
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
from datetime import datetime
from deep_translator import GoogleTranslator
from selenium.common.exceptions import TimeoutException, NoSuchElementException

In [ ]:
# scraping

def translate_text(text, source='en', target='fr'):
    try:
        return GoogleTranslator(source=source, target=target).translate(text)
    except:
        return text

In [ ]:
# parse the dates

def format_date(date_str):
    try:
       
        date_str = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', date_str.strip())

        parsed = datetime.strptime(date_str, "%B %d, %Y")
        return parsed.strftime("%d-%m-%Y")
    except Exception as e:
        print("Date parsing error:", date_str, "|", e)
        return date_str

In [ ]:
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36"

options = webdriver.ChromeOptions()
options.add_argument(f"user-agent={user_agent}")

driver = webdriver.Chrome(options=options)
tenders=[]

In [ ]:
driver.get("LINK") #add link
time.sleep(3)
wait = WebDriverWait(driver, 10)

In [ ]:
# scraping

while True:
    wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "tbody tr")))
    rows = driver.find_elements(By.CSS_SELECTOR, "tbody tr")
    if not rows:
        break  
    for i in range(len(rows)):
        rows = driver.find_elements(By.CSS_SELECTOR, "tbody tr") 
        row = rows[i]
        cols = row.find_elements(By.TAG_NAME, "td")
        if len(cols) < 10:
            continue
        
        raw_title = cols[1].text.strip()
        translated_title = translate_text(raw_title, source='en', target='fr')
        title = translated_title.lower()

        expiration_date_raw = cols[5].text.strip()
        publish_date_raw = cols[7].text.strip()

        expiration_date = format_date(expiration_date_raw)
        publish_date = format_date(publish_date_raw)

        actions_button = cols[9].find_element(By.TAG_NAME, "button")
        driver.execute_script("arguments[0].click();", actions_button)

        try:
            view_more = wait.until(EC.element_to_be_clickable(
                (By.XPATH, "//div[contains(@class,'v-overlay')]//div[contains(text(),'View more details')]")
            ))
            view_more.click()
            time.sleep(2)

            public_link_element = wait.until(
                EC.presence_of_element_located(
                    (By.XPATH, "//span[.='Public Link']/following-sibling::hr/following-sibling::span//a")
                )
            )
            link = public_link_element.get_attribute("href")

            tenders.append({
                "title": translated_title,
                "publication_date": publish_date,
                "expiration_date": expiration_date,
                "link": link,
                "country": "Kenya"
            })

            close_button = wait.until(EC.element_to_be_clickable(
                (By.XPATH, "//button[.//span[text()='Close']]")
            ))
            close_button.click()
            wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "tbody tr")))
            time.sleep(1)

        except (TimeoutException, NoSuchElementException):
            print("Button not found")
          
            try:
                close_button = wait.until(EC.element_to_be_clickable(
                    (By.XPATH, "//button[.//span[text()='Close']]")
                ))
                close_button.click()
                wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "tbody tr")))
                time.sleep(1)
            except:
                pass
            continue

    try:
        next_button = driver.find_element(By.CSS_SELECTOR, "li.v-pagination__next button")
        if 'v-btn--disabled' in next_button.get_attribute('class'):
            print("end of pagination")
            break 
        else:
            driver.execute_script("arguments[0].scrollIntoView(true);", next_button)
            next_button.click()
            time.sleep(2)  
    except Exception as e:
        print("Next button error or end of pagination:", e)
        break

In [ ]:
driver.quit()

In [ ]:
def clean_excel_string(s):
    if isinstance(s, str):
        return re.sub(r'[\x00-\x1F\x7F-\x9F]', '', s)
    return s

df = pd.DataFrame(tenders)
df = df.applymap(clean_excel_string)

df.to_excel("../2_scraped_files/Kenya.xlsx", index=False) # modify